In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

In [3]:
from pinecone import Pinecone
pc = Pinecone(api_key=api_key)

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)

d:\RAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5799.38it/s]


In [5]:
from pinecone import ServerlessSpec

index_name ="rag" #

if not pc.has_index(index_name):
    index =pc.create_index(
    name="rag",
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)
    
index = pc.Index(index_name)

index

In [6]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [7]:
index = pc.Index("rag")
index

print(index)
print(type(index))

<class 'pinecone.db_data.index.Index'>


In [8]:
vector_store

In [9]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [10]:
vector_store.add_documents(documents=documents)

['0a881afc-1538-4972-ba95-82fa9dea08e4',
 '9da5b790-9a30-4073-bca8-dafb61bf2fb8',
 '9260cbe0-5ad7-48de-bc6d-775ac9054efb',
 '495ed01c-6f15-446c-b8a6-dbeaa497171f',
 '82aa3713-f75d-4bad-b699-2fe4550afd8b',
 'ea742924-e3d7-467b-8fb0-cd4f6879ccae',
 '777c2e4f-a616-4629-8ca1-2a320482ef09',
 'c80a89d7-64cb-484a-a362-dd3a7b4710ce',
 '5c82fb02-73d3-419e-b62b-076c256ef855',
 '3765656a-8b87-49b2-b4cc-b91b25f9358e']

In [11]:
### Query Directly

result = vector_store.similarity_search(
    "Langchain provide abstractions to make working with LLMs easy",
    k=2,
    filter = {"source":"tweet"}
)

for res in result:
    print(f'* "{res.page_content}", metadata={res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}


In [12]:
result = vector_store.similarity_search_with_score(
    "Langchain provide abstractions to make working with LLMs easy",
    k=3,
    filter = {"source":"tweet"}
)

for res, score in result:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.58] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.46] "LangGraph is the best framework for building stateful, agentic applications!", metadata={'source': 'tweet'}
* [SIM=0.02] "I have a bad feeling I am going to get deleted :(", metadata={'source': 'tweet'}


In [14]:
result = vector_store.similarity_search_with_score(
    "will it ne hot tomorrow?",
    k=1,
    filter = {"source":"news"}
)

for res, score in result:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.50] "The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.", metadata={'source': 'news'}


In [15]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.4
    }
)
retriever.invoke("Stealing from the bank is crime", filter={"source": "news"})

[Document(id='495ed01c-6f15-446c-b8a6-dbeaa497171f', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [16]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 1
    },
        
)
retriever.invoke("Stealing from the bank is crime", filter={"source": "news"})

[Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]